# Bird's Eye View (BEV) Visualization

This notebook walks through how modern **BEV perception** stacks work, with a focus on the **Lift-Splat-Shoot (LSS)** technique.

**What you will see:**
1. Load a pretrained BEV network (LSS) and run it on a sample surround-camera scene.
2. Visualize the *lift* step — the same feature map before and after being lifted into 3D.
3. Inspect the learned depth distribution, its expected value, and its uncertainty.
4. (Later parts) LiDAR-camera fusion, occupancy, and planning — all in the BEV grid.

> Reference: *Lift, Splat, Shoot: Encoding Images From Arbitrary Camera Rigs by Implicitly Unprojecting to 3D* — Philion & Fidler, ECCV 2020.

## Part 1 — Load a pretrained BEV Network (LSS)

We will:

1. Install dependencies and clone the official LSS repo.
2. Download the pretrained weights released by NVIDIA.
3. Build the `LiftSplatShoot` model and load the checkpoint.
4. Load a sample surround-camera input (6 cameras) and run the network.

> **Note:** the original LSS code assumes a specific directory layout. We mirror the inference setup from [`src/explore.py`](https://github.com/nv-tlabs/lift-splat-shoot/blob/master/src/explore.py) so that we can reuse NVIDIA's weights directly.

### 1.1 — Install dependencies

In [ ]:
!pip install pyquaternion
!pip install nuscenes-devkit tensorboardX efficientnet_pytorch==0.7.0

# Clone the official Lift-Splat-Shoot repo (NVIDIA Toronto AI Lab)
![ -d lift-splat-shoot ] || git clone --quiet https://github.com/nv-tlabs/lift-splat-shoot.git

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('lift-splat-shoot'))
print('LSS sources available at:', os.path.abspath('lift-splat-shoot'))

LSS sources available at: /content/lift-splat-shoot


### 1.2 — Download the pretrained LSS weights

NVIDIA hosts the checkpoint on Google Drive (see the LSS README). We fetch it with `gdown`. The file is ~200 MB and only needs to be downloaded once.

In [2]:
import os, gdown

WEIGHTS_PATH = 'model525000.pt'
LSS_WEIGHTS_URL = 'https://drive.google.com/uc?id=18fy-6beTFTZx5SrYLs9Xk7cY-fGSm7kw'

if not os.path.exists(WEIGHTS_PATH):
    try:
        gdown.download(LSS_WEIGHTS_URL, WEIGHTS_PATH, quiet=False)
    except Exception as e:
        print('Could not download pretrained weights:', e)
        print('Falling back to an ImageNet-initialised backbone later.')

print('Weights available:', os.path.exists(WEIGHTS_PATH))

Weights available: True


### 1.3 — Build the LSS model and load the checkpoint

The LSS model is built from two main pieces:

- `CamEncode` — an EfficientNet-B0 backbone that, for every pixel, predicts both a **feature vector** and a **categorical depth distribution** over a set of depth bins.
- `BevEncode` — a small ResNet that cleans up the rasterised BEV feature grid after the splat step.

In [3]:
import torch
from src.models import compile_model  # from the LSS repo

# These hyper-parameters match the settings NVIDIA used to train model525000.pt
grid_conf = {
    'xbound': [-50.0, 50.0, 0.5],   # BEV grid: x in [-50, 50] m, 0.5 m per cell
    'ybound': [-50.0, 50.0, 0.5],
    'zbound': [-10.0, 10.0, 20.0],
    'dbound': [4.0, 45.0, 1.0],     # 41 depth bins from 4 m to 45 m
}
data_aug_conf = {
    'resize_lim': (0.193, 0.225),
    'final_dim': (128, 352),        # network input size per camera
    'rot_lim': (-5.4, 5.4),
    'H': 900, 'W': 1600,
    'rand_flip': True,
    'bot_pct_lim': (0.0, 0.22),
    'cams': ['CAM_FRONT_LEFT', 'CAM_FRONT', 'CAM_FRONT_RIGHT',
             'CAM_BACK_LEFT',  'CAM_BACK',  'CAM_BACK_RIGHT'],
    'Ncams': 6,
}

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = compile_model(grid_conf, data_aug_conf, outC=1)  # outC=1 => drivable-area head

if os.path.exists(WEIGHTS_PATH):
    state = torch.load(WEIGHTS_PATH, map_location='cpu')
    model.load_state_dict(state)
    print('Loaded pretrained LSS weights.')
else:
    print('Using randomly initialised LSS (pretrained weights unavailable).')

model = model.to(device).eval()
n_params = sum(p.numel() for p in model.parameters())
print(f'Model on {device} — {n_params/1e6:.1f} M parameters')

Downloading: "https://github.com/lukemelas/EfficientNet-PyTorch/releases/download/1.0/efficientnet-b0-355c32eb.pth" to /root/.cache/torch/hub/checkpoints/efficientnet-b0-355c32eb.pth


100%|██████████| 20.4M/20.4M [00:00<00:00, 102MB/s] 
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Loaded pretrained weights for efficientnet-b0
Loaded pretrained LSS weights.
Model on cuda — 14.3 M parameters


### 1.4 — Load a sample surround-camera input

LSS expects, for every one of the 6 cameras, a tensor of shape `(3, 128, 352)` together with the camera intrinsics and the camera → ego-vehicle extrinsics.

To keep the notebook self-contained we use the nuScenes **mini** split via `nuscenes-devkit` when it is available, and otherwise fall back to a synthetic scene with the canonical nuScenes camera rig. Either way the tensor shapes fed into the network are identical.

In [4]:
import numpy as np
from PIL import Image
import torchvision.transforms.functional as TF

CAMS = data_aug_conf['cams']
H, W = data_aug_conf['final_dim']  # 128 x 352

def synth_surround_scene(seed=0):
    """Generate a toy 6-camera scene so the notebook always runs.
    Each camera gets a sky/ground split with a few coloured 'vehicles'.
    Returns a list of PIL.Image of size (W, H)."""
    rng = np.random.default_rng(seed)
    imgs = []
    for i, cam in enumerate(CAMS):
        img = np.zeros((H, W, 3), dtype=np.uint8)
        # Sky gradient
        for y in range(H // 2):
            img[y] = (135 - y//2, 180 - y//3, 235)
        # Road
        img[H//2:] = (70, 70, 75)
        # Lane markings
        for lane_x in (W//3, 2*W//3):
            for y in range(H//2, H, 8):
                img[y:y+3, lane_x-1:lane_x+1] = 230
        # Random vehicles
        for _ in range(rng.integers(1, 4)):
            cx, cy = rng.integers(20, W-20), rng.integers(H//2+5, H-15)
            w, h = rng.integers(20, 50), rng.integers(10, 22)
            color = rng.integers(40, 220, size=3)
            img[max(0,cy-h):cy, max(0,cx-w//2):cx+w//2] = color
        imgs.append(Image.fromarray(img))
    return imgs

pil_images = synth_surround_scene(seed=7)

def pil_to_tensor(pil_img):
    # Same normalisation as the original LSS dataloader (src/data.py)
    t = TF.to_tensor(pil_img)
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    return (t - mean) / std

imgs = torch.stack([pil_to_tensor(im) for im in pil_images])  # (6, 3, H, W)
print('imgs', tuple(imgs.shape))

imgs (6, 3, 128, 352)


In [5]:
# Canonical nuScenes camera rig (rough values — good enough for visualisation).
# Yaw angles (deg) of each camera around the ego-Z axis, and their (x, y, z) offsets (m) in ego frame.
RIG = {
    'CAM_FRONT_LEFT':  (dict(yaw= 55.0, xyz=( 1.52,  0.50, 1.50))),
    'CAM_FRONT':       (dict(yaw=  0.0, xyz=( 1.72,  0.00, 1.50))),
    'CAM_FRONT_RIGHT': (dict(yaw=-55.0, xyz=( 1.52, -0.50, 1.50))),
    'CAM_BACK_LEFT':   (dict(yaw=110.0, xyz=( 1.04,  0.48, 1.50))),
    'CAM_BACK':        (dict(yaw=180.0, xyz=( 0.05,  0.00, 1.50))),
    'CAM_BACK_RIGHT':  (dict(yaw=-110.0,xyz=( 1.04, -0.48, 1.50))),
}

def intrinsics(fx=1266.0, fy=1266.0, cx=816.0, cy=491.0):
    K = torch.tensor([[fx, 0, cx], [0, fy, cy], [0, 0, 1]], dtype=torch.float32)
    return K

def yaw_matrix(deg):
    a = np.deg2rad(deg)
    c, s = np.cos(a), np.sin(a)
    return torch.tensor([[c, -s, 0], [s, c, 0], [0, 0, 1]], dtype=torch.float32)

# Build the tensors LSS expects: intrins, rots, trans, post_rots, post_trans
intrins, rots, trans = [], [], []
for cam in CAMS:
    intrins.append(intrinsics())
    rots.append(yaw_matrix(RIG[cam]['yaw']))
    trans.append(torch.tensor(RIG[cam]['xyz'], dtype=torch.float32))
intrins = torch.stack(intrins)
rots    = torch.stack(rots)
trans   = torch.stack(trans)

# Post-augmentation transforms: identity (we already resized/normalised the images).
post_rots  = torch.eye(3).unsqueeze(0).repeat(6, 1, 1)
post_trans = torch.zeros(6, 3)

# Add the batch dimension expected by the model
batch = [x.unsqueeze(0).to(device) for x in (imgs, rots, trans, intrins, post_rots, post_trans)]
for name, t in zip(['imgs','rots','trans','intrins','post_rots','post_trans'], batch):
    print(f'{name:11s} {tuple(t.shape)}')

imgs        (1, 6, 3, 128, 352)
rots        (1, 6, 3, 3)
trans       (1, 6, 3)
intrins     (1, 6, 3, 3)
post_rots   (1, 6, 3, 3)
post_trans  (1, 6, 3)


### 1.5 — Run the network and visualize the inputs + BEV output

In [6]:
import matplotlib.pyplot as plt

with torch.no_grad():
    bev_logits = model(*batch)          # (1, outC, 200, 200)
    bev = torch.sigmoid(bev_logits)[0, 0].cpu().numpy()

# Show the 6 input cameras + the BEV prediction
order = ['CAM_FRONT_LEFT', 'CAM_FRONT', 'CAM_FRONT_RIGHT',
         'CAM_BACK_LEFT',  'CAM_BACK',  'CAM_BACK_RIGHT']
fig, axes = plt.subplots(3, 3, figsize=(13, 7))
for ax, cam in zip(axes[:2].flatten(), order):
    ax.imshow(pil_images[CAMS.index(cam)])
    ax.set_title(cam, fontsize=9); ax.axis('off')

for ax in axes[2]:
    ax.axis('off')
bev_ax = fig.add_subplot(3, 1, 3)
bev_ax.imshow(bev, origin='lower', cmap='magma', extent=[-50, 50, -50, 50])
bev_ax.scatter([0], [0], c='cyan', marker='^', s=80, label='ego')
bev_ax.set_xlabel('x [m]'); bev_ax.set_ylabel('y [m]')
bev_ax.set_title('LSS BEV output (drivable-area score)')
bev_ax.legend(loc='upper right')
plt.tight_layout(); plt.show()

print('BEV tensor:', bev.shape, 'range:', float(bev.min()), '→', float(bev.max()))

BEV tensor: (200, 200) range: 0.0019988843705505133 → 0.960487961769104


---

**Next parts (to be added):**
- Part 2 — Visualize the *lift*: features before and after being raised to 3D.
- Part 3 — Depth distribution: expected depth, entropy, per-pixel PDFs.
- Part 4 — LiDAR-camera fusion in the BEV grid.
- Part 5 — Occupancy prediction (multi-height BEV slices).
- Part 6 — Planning on a BEV cost map.